# 🤖 Sistema Multi-Agente TechCorp

## RAG con Enrutamiento Inteligente + Observabilidad con Langfuse

Este notebook implementa un sistema de múltiples agentes especializados que responden
consultas corporativas de tres dominios:

| Agente | Dominio | Base de Conocimiento |
|--------|---------|---------------------|
| 🧑‍💼 HR Agent | Recursos Humanos | Políticas, beneficios, vacaciones, desempeño |
| 💻 Tech Agent | Tecnología / IT | Infraestructura, dev, seguridad, troubleshooting |
| 💰 Finance Agent | Finanzas | Gastos, reembolsos, presupuesto, proveedores |

**Flujo del sistema:**
```
Usuario → Orquestador (clasificación LLM) → Agente Especializado (RAG) → Respuesta
                                                    ↓
                                               Langfuse (observabilidad)
```

---
**Orden de ejecución:**
1. Sección 1: Setup e Imports
2. Sección 2: Carga de Documentos y Vector Stores
3. Sección 3: Definición de Agentes
4. Sección 4: Orquestador y Enrutamiento
5. Sección 5: Pruebas y Ejemplos
6. Sección 6: Integración con Langfuse
7. Sección 7 (BONUS): Agente Evaluador

---
## 📦 Sección 1: Setup e Imports

Configuración del entorno, instalación de dependencias y carga de variables de entorno.

In [1]:
# Instalar dependencias si no están instaladas
# Descomenta y ejecuta si es la primera vez
# !pip install -r requirements.txt

In [2]:
import os
import sys
import json
import math
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

# Agregar src al path
BASE_DIR = Path(".").resolve()
SRC_DIR = BASE_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

from dotenv import load_dotenv
from openai import OpenAI

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que las API keys estén configuradas
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

if not OPENAI_API_KEY:
    raise ValueError("❌ OPENAI_API_KEY no configurada. Crea un archivo .env basado en .env.example")

print("✅ Variables de entorno cargadas")
print(f"   OpenAI API Key: {'✓ configurada' if OPENAI_API_KEY else '✗ faltante'}")
print(f"   Langfuse Public Key: {'✓ configurada' if LANGFUSE_PUBLIC_KEY else '⚠ no configurada (opcional)'}")
print(f"   Langfuse Secret Key: {'✓ configurada' if LANGFUSE_SECRET_KEY else '⚠ no configurada (opcional)'}")

✅ Variables de entorno cargadas
   OpenAI API Key: ✓ configurada
   Langfuse Public Key: ✓ configurada
   Langfuse Secret Key: ✓ configurada


In [3]:
# Inicializar cliente de OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Verificar conexión
test_response = openai_client.models.list()
print(f"✅ OpenAI conectado. Modelos disponibles: {len(list(test_response.data))}")

# Configuración de modelos
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
ROUTING_MODEL = os.getenv("ROUTING_MODEL", "gpt-4o-mini")

print(f"\n📋 Modelos configurados:")
print(f"   Embedding: {EMBEDDING_MODEL}")
print(f"   Chat/RAG:  {CHAT_MODEL}")
print(f"   Routing:   {ROUTING_MODEL}")

✅ OpenAI conectado. Modelos disponibles: 119

📋 Modelos configurados:
   Embedding: text-embedding-3-small
   Chat/RAG:  gpt-4o-mini
   Routing:   gpt-4o-mini


---
## 📚 Sección 2: Carga de Documentos y Vector Stores

Esta sección carga los documentos de cada dominio, los divide en chunks,
genera embeddings y construye los vector stores.

**Estructura de documentos:**
```
data/
├── hr_docs/          → 3 archivos con políticas de RRHH
├── tech_docs/        → 3 archivos con documentación técnica
└── finance_docs/     → 3 archivos con políticas financieras
```

In [4]:
# Importar el módulo principal del sistema
from multi_agent_system import (
    load_domain_docs,
    generate_embeddings,
    save_vector_store,
    load_vector_store,
    build_all_vector_stores,
    DATA_DIR,
    OUTPUTS_DIR,
)

# Mostrar documentos disponibles por dominio
print("📁 Documentos disponibles por dominio:")
for domain in ["hr_docs", "tech_docs", "finance_docs"]:
    domain_dir = DATA_DIR / domain
    if domain_dir.exists():
        files = list(domain_dir.glob("*"))
        total_chars = sum(f.read_text(encoding='utf-8').__len__() for f in files if f.is_file())
        print(f"   {domain}/: {len(files)} archivos, ~{total_chars:,} caracteres")
        for f in files:
            print(f"      - {f.name}")
    else:
        print(f"   ⚠ {domain}/ no encontrado")

📁 Documentos disponibles por dominio:
   hr_docs/: 4 archivos, ~30,732 caracteres
      - faq_rrhh.txt
      - guia_reclutamiento_capacitacion.txt
      - politicas_generales.txt
      - reglamento_interno.txt
   tech_docs/: 4 archivos, ~32,538 caracteres
      - it_knowledge_base.txt
      - devops_cloud_guide.txt
      - troubleshooting_faq.txt
      - arquitectura_estandares.txt
   finance_docs/: 4 archivos, ~32,004 caracteres
      - planificacion_metricas_controles.txt
      - contabilidad_inversiones.txt
      - faq_finanzas.txt
      - politicas_gastos.txt


In [5]:
# Construir todos los vector stores
#
# ✅ FIRMA CORRECTA:
#    build_all_vector_stores(client=openai_client, force_rebuild=False)
#
# ❌ ERRORES COMUNES:
#    client=OPENAI_API_KEY   → pasar el string de la key en vez del objeto OpenAI
#    embeddin=...            → argumento inexistente (typo de 'embedding_model')
#    El modelo de embedding se configura en .env (EMBEDDING_MODEL), no aquí.
#
# La primera vez llama a la API de embeddings de OpenAI (tiene costo).
# Las veces siguientes carga desde disco (sin costo adicional).
# Usa force_rebuild=True solo si modificaste los documentos en data/.

# Validación defensiva antes de llamar
from openai import OpenAI as _OpenAIClass
assert isinstance(openai_client, _OpenAIClass), (
    "❌ 'openai_client' debe ser una instancia de OpenAI, no un string. "
    "Usa: openai_client = OpenAI(api_key=OPENAI_API_KEY)"
)

vector_stores = build_all_vector_stores(
    client=openai_client,       # ← instancia OpenAI, NO el string de la API key
    force_rebuild=False,        # ← True para reconstruir índices desde cero
)

# Verificar mínimo de 50 chunks por dominio
print("\n📊 Validación de chunks por dominio:")
for domain, vs in vector_stores.items():
    n = len(vs["chunks"])
    status = "✅" if n >= 50 else "⚠"
    print(f"   {status} {domain}: {n} chunks {'(OK)' if n >= 50 else '(mínimo 50 requerido)'}")


🔨 HR: Construyendo vector store desde /home/antoniozl/Documents/multi_agent_system/data/hr_docs
   ✓ faq_rrhh.txt: 14 chunks
   ✓ guia_reclutamiento_capacitacion.txt: 12 chunks
   ✓ politicas_generales.txt: 21 chunks
   ✓ reglamento_interno.txt: 19 chunks
   → 66 chunks en total, generando embeddings...
   ✓ Guardado en /home/antoniozl/Documents/multi_agent_system/outputs/hr_index.json (2.07 MB)
   ✅ HR listo: 66 chunks indexados

🔨 TECH: Construyendo vector store desde /home/antoniozl/Documents/multi_agent_system/data/tech_docs
   ✓ it_knowledge_base.txt: 17 chunks
   ✓ devops_cloud_guide.txt: 17 chunks
   ✓ troubleshooting_faq.txt: 15 chunks
   ✓ arquitectura_estandares.txt: 16 chunks
   → 65 chunks en total, generando embeddings...
   ✓ Guardado en /home/antoniozl/Documents/multi_agent_system/outputs/tech_index.json (2.16 MB)
   ✅ TECH listo: 65 chunks indexados

🔨 FINANCE: Construyendo vector store desde /home/antoniozl/Documents/multi_agent_system/data/finance_docs
   ✓ planifica

In [6]:
# Inspeccionar un chunk de ejemplo de cada dominio
print("🔍 Muestra de chunks por dominio:\n")
for domain, vs in vector_stores.items():
    sample = vs["chunks"][5]  # Chunk #5 como muestra
    print(f"── {domain.upper()} (chunk #{sample['chunk_id']}, source: {sample.get('source', 'N/A')}) ──")
    print(sample["text"][:250] + "...")
    print(f"   Chars: {sample['char_count']} | Embedding dim: {len(sample['embedding'])}\n")

🔍 Muestra de chunks por dominio:

── HR (chunk #5, source: faq_rrhh.txt) ──
**¿Qué hago si me enfermo y no puedo ir a trabajar?**
Notifica a tu manager directo lo antes posible (preferiblemente antes de las 9:00 AM) por Slack o correo. Si la ausencia dura más de 1 día, sube el certificado médico a Workday > Tiempo y Ausencia...
   Chars: 598 | Embedding dim: 1536

── TECH (chunk #5, source: it_knowledge_base.txt) ──
**Diseño y Producto:**
- MacBook Pro 14" (M3, 16GB RAM, 512GB SSD)
- iPad Pro con Apple Pencil (si aplica al rol)
- Monitor externo 27" con soporte de color calibrado
**Administración y Negocio:**
- MacBook Air 13" o 15" (M3, 16GB RAM, 256GB SSD)
- M...
   Chars: 574 | Embedding dim: 1536

── FINANCE (chunk #5, source: planificacion_metricas_controles.txt) ──
---
## PARTE 2: MÉTRICAS SAAS Y ANÁLISIS DE NEGOCIO
### 2.1 Métricas de Revenue
**MRR (Monthly Recurring Revenue):**
El MRR es la métrica más importante para un negocio SaaS. Se calcula como:
`MRR = Suma de todos los i

---
## 🤖 Sección 3: Definición de Agentes

Inicialización de los tres agentes especializados. Cada agente:
- Tiene su propio vector store con conocimiento de dominio
- Usa un system prompt optimizado para su área
- Implementa búsqueda vectorial + generación con RAG

In [7]:
from agents.hr_agent import HRAgent
from agents.tech_agent import TechAgent
from agents.finance_agent import FinanceAgent

# Inicializar agentes con sus respectivos vector stores
hr_agent = HRAgent(
    openai_client=openai_client,
    vector_store=vector_stores["hr"],
    top_k=4,
)

tech_agent = TechAgent(
    openai_client=openai_client,
    vector_store=vector_stores["tech"],
    top_k=4,
)

finance_agent = FinanceAgent(
    openai_client=openai_client,
    vector_store=vector_stores["finance"],
    top_k=4,
)

print("✅ Agentes inicializados:")
print(f"   🧑‍💼 {hr_agent.name}: {len(vector_stores['hr']['chunks'])} chunks")
print(f"   💻 {tech_agent.name}: {len(vector_stores['tech']['chunks'])} chunks")
print(f"   💰 {finance_agent.name}: {len(vector_stores['finance']['chunks'])} chunks")

✅ Agentes inicializados:
   🧑‍💼 Agente de Recursos Humanos: 66 chunks
   💻 Agente de Soporte Técnico: 65 chunks
   💰 Agente de Finanzas: 68 chunks


In [8]:
# Prueba directa del HR Agent (sin orquestador)
print("🧪 Prueba directa — HR Agent:\n")

hr_result = hr_agent.answer("¿Cuántos días de vacaciones tengo al año?")

print(f"Pregunta: {hr_result['question']}")
print(f"Confianza: {hr_result['confidence_score']}")
print(f"\nRespuesta:\n{hr_result['answer']}")
print(f"\nChunks utilizados: {[c['chunk_id'] for c in hr_result['chunks_used']]}")

🧪 Prueba directa — HR Agent:

Pregunta: ¿Cuántos días de vacaciones tengo al año?
Confianza: 0.5903

Respuesta:
En TechCorp, tienes derecho a 15 días de vacaciones al año. Estos días se acumulan mensualmente a razón de 1.25 días por mes. Puedes verificar tu saldo actual en Workday, en la sección "Mi Tiempo" > "Balance de Ausencias". Si necesitas más información, no dudes en preguntar.

Chunks utilizados: [36, 3, 35, 4]


In [9]:
# Prueba directa del Tech Agent (sin orquestador)
print("🧪 Prueba directa — Tech Agent:\n")

tech_result = tech_agent.answer("¿Cómo configuro la VPN en mi computador?")

print(f"Pregunta: {tech_result['question']}")
print(f"Confianza: {tech_result['confidence_score']}")
print(f"\nRespuesta:\n{tech_result['answer']}")

🧪 Prueba directa — Tech Agent:

Pregunta: ¿Cómo configuro la VPN en mi computador?
Confianza: 0.5036

Respuesta:
Para configurar la VPN en tu computador, sigue estos pasos:

1. **Descarga el cliente**: Ve a it.techcorp.com/vpn y descarga el cliente de Cisco AnyConnect.
2. **Instala el cliente**: Ejecuta el archivo descargado y sigue las instrucciones del wizard de instalación.
3. **Configura el servidor**: Abre la aplicación Cisco AnyConnect e ingresa la dirección del servidor: `vpn.techcorp.com`.
4. **Autenticación**: Ingresa tus credenciales de Google y el código de autenticación de dos factores (2FA). Asegúrate de que el código sea el más reciente, ya que expira cada 30 segundos.

Si necesitas más ayuda, puedes contactar al soporte a través de email a it-support@techcorp.com, en Slack en el canal #it-support, o llamando a la extensión 3000.


In [10]:
# Prueba directa del Finance Agent (sin orquestador)
print("🧪 Prueba directa — Finance Agent:\n")

finance_result = finance_agent.answer("¿Cómo pido el reembolso de un gasto de viaje?")

print(f"Pregunta: {finance_result['question']}")
print(f"Confianza: {finance_result['confidence_score']}")
print(f"\nRespuesta:\n{finance_result['answer']}")

🧪 Prueba directa — Finance Agent:

Pregunta: ¿Cómo pido el reembolso de un gasto de viaje?
Confianza: 0.5464

Respuesta:
Para solicitar el reembolso de un gasto de viaje, sigue estos pasos:

1. **Accede a Expensify**: Descarga la app Expensify o ingresa a app.expensify.com.
2. **Sube el comprobante**: Toma una foto del comprobante del gasto o sube el PDF correspondiente.
3. **Completa la información**: Asegúrate de ingresar todos los detalles del gasto, incluyendo la fecha, el monto y la categoría correspondiente.
4. **Envía la solicitud**: Una vez que hayas ingresado toda la información, envía la solicitud de reembolso a través de la plataforma.

Recuerda que los gastos deben ser reportados con su comprobante correspondiente. Si el gasto es menor a USD 25 y no tienes el comprobante, puedes utilizar una declaración jurada. Para montos mayores, necesitarás el comprobante para que el gasto sea reembolsable.


---
## 🎯 Sección 4: Orquestador y Enrutamiento

El orquestador recibe la pregunta del usuario, la clasifica mediante un LLM
de routing y delega al agente especializado correspondiente.

**Estrategia de routing:**
- Se usa GPT como clasificador zero-shot con output JSON estructurado
- El routing es determinístico (`temperature=0`)
- Si la confianza es < 0.4, se trata como dominio desconocido

In [11]:
from agents.orchestrator import Orchestrator

# Inicializar el orquestador (sin Langfuse por ahora)
orchestrator = Orchestrator(
    openai_client=openai_client,
    hr_agent=hr_agent,
    tech_agent=tech_agent,
    finance_agent=finance_agent,
    langfuse_client=None,  # Se agregará Langfuse en la Sección 6
)

print("✅ Orquestador inicializado")
print("   Agentes disponibles:", list(orchestrator.agents.keys()))

✅ Orquestador inicializado
   Agentes disponibles: ['hr', 'tech', 'finance']


In [12]:
# Función helper para mostrar resultados de forma legible
def print_result(result: dict, show_chunks: bool = False):
    routing = result.get('routing', {})
    print(f"{'='*60}")
    print(f"❓ Pregunta:   {result.get('question', '')}")
    print(f"🎯 Dominio:    {routing.get('domain', result.get('domain', 'N/A'))} "
          f"(confianza: {routing.get('confidence', 'N/A')})")
    print(f"🔍 Razonamiento: {routing.get('reasoning', 'N/A')}")
    print(f"🤖 Agente:     {result.get('agent', 'N/A')}")
    print(f"📊 Score RAG:  {result.get('confidence_score', 'N/A')}")
    print(f"\n💬 Respuesta:\n{result.get('answer', '')}")
    if show_chunks and result.get('chunks_used'):
        print(f"\n📎 Chunks utilizados:")
        for c in result['chunks_used']:
            print(f"   - ID:{c['chunk_id']} | sim:{c['similarity']} | {c.get('source','?')}")
    print(f"{'='*60}\n")

In [ ]:
# Demostración del routing automático — consulta de RRHH
result_hr = orchestrator.route_and_answer(
    "¿puedo acceder a la copia de seguridad de mi usuario?"
)
print_result(result_hr)

❓ Pregunta:   ¿Cuál es la política de licencia de maternidad?
🎯 Dominio:    hr (confianza: 0.9)
🔍 Razonamiento: La pregunta se refiere a una política de Recursos Humanos relacionada con licencias.
🤖 Agente:     Agente de Recursos Humanos
📊 Score RAG:  0.5154

💬 Respuesta:
Lamentablemente, la información sobre la política de licencia de maternidad no está incluida en el contexto proporcionado. Te sugiero que contactes directamente al departamento de Recursos Humanos para obtener detalles específicos sobre este tema. Estoy aquí para ayudarte con cualquier otra pregunta que tengas.



In [ ]:
# Demostración del routing automático — consulta técnica
result_tech = orchestrator.route_and_answer(
    "¿Qué estándares debo seguir para hacer commits en GitHub?"
)
print_result(result_tech)

In [ ]:
# Demostración del routing automático — consulta financiera
result_finance = orchestrator.route_and_answer(
    "¿Qué documentos necesito para que un proveedor sea dado de alta?"
)
print_result(result_finance)

In [ ]:
# Demostración del manejo de dominio desconocido
result_unknown = orchestrator.route_and_answer(
    "¿Cuál es el mejor restaurante italiano en Madrid?"
)
print_result(result_unknown)

---
## 🧪 Sección 5: Pruebas y Ejemplos

Ejecución del conjunto de test queries para validar el sistema.
Se prueban los 18 casos definidos en `test_queries.json`, incluyendo
casos de cada dominio y casos borde.

In [ ]:
# Cargar test queries
TEST_QUERIES_PATH = BASE_DIR / "test_queries.json"

with open(TEST_QUERIES_PATH, "r", encoding="utf-8") as f:
    test_queries = json.load(f)

print(f"✅ {len(test_queries)} test queries cargadas")
print("\nDistribución por dominio esperado:")

from collections import Counter
domain_counts = Counter(q["expected_domain"] for q in test_queries)
for domain, count in sorted(domain_counts.items()):
    print(f"   {domain}: {count} queries")

In [ ]:
# Ejecutar TODAS las test queries
print("🚀 Ejecutando test queries...\n")

test_results = []
routing_correct = 0
routing_total = 0

for i, tq in enumerate(test_queries, 1):
    print(f"[{i:02d}/{len(test_queries)}] {tq['query'][:70]}...")
    
    result = orchestrator.route_and_answer(tq["query"])
    result["expected_domain"] = tq["expected_domain"]
    result["category"] = tq.get("category", "")
    test_results.append(result)
    
    # Evaluar routing
    predicted = result.get("routing", {}).get("domain", result.get("domain", ""))
    expected = tq["expected_domain"]
    is_correct = predicted == expected
    routing_correct += int(is_correct)
    routing_total += 1
    
    status = "✅" if is_correct else "❌"
    print(f"   {status} Routing: {expected} → {predicted}\n")

routing_accuracy = routing_correct / routing_total
print(f"\n{'='*50}")
print(f"📊 Routing Accuracy: {routing_correct}/{routing_total} = {routing_accuracy:.1%}")
print(f"{'='*50}")

In [ ]:
# Análisis detallado de resultados
print("📋 Resumen de Test Results:\n")

routing_errors = [
    r for r in test_results
    if r.get("routing", {}).get("domain", r.get("domain")) != r["expected_domain"]
]

if routing_errors:
    print(f"⚠ Errores de routing ({len(routing_errors)}):")
    for r in routing_errors:
        predicted = r.get("routing", {}).get("domain", r.get("domain"))
        print(f"   - '{r['question'][:60]}...'")
        print(f"     Esperado: {r['expected_domain']} | Obtenido: {predicted}")
else:
    print("🎉 ¡Todos los routings fueron correctos!")

print(f"\n📊 Confidence Scores promedio por dominio:")
for domain in ["hr", "tech", "finance"]:
    domain_results = [r for r in test_results if r.get("domain") == domain]
    if domain_results:
        avg_conf = sum(r.get("confidence_score", 0) for r in domain_results) / len(domain_results)
        print(f"   {domain}: {avg_conf:.4f} (n={len(domain_results)})")

In [ ]:
# Mostrar respuestas de los casos borde
print("🔍 Casos borde — Análisis:\n")

edge_cases = [
    "¿El clima en Bogotá hoy?",
    "¿Qué pasa si un proveedor me pide un pago en efectivo para agilizar algo?",
    "¿Puedo llevar a mi pareja al plan de seguro médico si no estamos casados?",
]

for r in test_results:
    if any(ec[:30] in r.get("question", "") for ec in edge_cases):
        print_result(r, show_chunks=False)

In [ ]:
# Guardar resultados de tests
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
results_path = OUTPUTS_DIR / "test_results.json"

# Limpiar embeddings antes de guardar (muy pesados)
results_to_save = []
for r in test_results:
    r_clean = {k: v for k, v in r.items() if k != "chunks_used"}
    r_clean["chunks_used_ids"] = [c["chunk_id"] for c in r.get("chunks_used", [])]
    results_to_save.append(r_clean)

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results_to_save, f, ensure_ascii=False, indent=2)

print(f"✅ Resultados guardados en: {results_path}")

---
## 🔭 Sección 6: Integración con Langfuse

Langfuse es una plataforma de observabilidad para LLMs que permite:
- **Trazabilidad**: Ver cada paso del pipeline (routing → RAG → generación)
- **Latencia**: Medir tiempos de cada etapa
- **Costos**: Estimar costos de tokens por consulta
- **Scores**: Registrar métricas de calidad de respuesta

> ⚠️ **Requiere configurar** `LANGFUSE_PUBLIC_KEY` y `LANGFUSE_SECRET_KEY` en `.env`.
> Si no tienes cuenta, regístrate gratis en https://cloud.langfuse.com

In [ ]:
# Inicializar Langfuse
langfuse_client = None

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    try:
        from langfuse import Langfuse
        
        langfuse_client = Langfuse(
            public_key=LANGFUSE_PUBLIC_KEY,
            secret_key=LANGFUSE_SECRET_KEY,
            host=LANGFUSE_HOST,
        )
        
        # Verificar conexión
        langfuse_client.auth_check()
        print("✅ Langfuse conectado correctamente")
        print(f"   Host: {LANGFUSE_HOST}")
        
    except Exception as e:
        print(f"⚠ Error conectando Langfuse: {e}")
        print("   El sistema continuará sin observabilidad.")
        langfuse_client = None
else:
    print("⚠ Langfuse no configurado. Agrega LANGFUSE_PUBLIC_KEY y LANGFUSE_SECRET_KEY al .env")
    print("   El sistema funciona correctamente sin Langfuse.")

In [ ]:
# Reinicializar el orquestador CON Langfuse
orchestrator_with_langfuse = Orchestrator(
    openai_client=openai_client,
    hr_agent=hr_agent,
    tech_agent=tech_agent,
    finance_agent=finance_agent,
    langfuse_client=langfuse_client,  # Puede ser None si no está configurado
)

print(f"✅ Orquestador con Langfuse: {'activo' if langfuse_client else 'inactivo (funciona sin Langfuse)'}")

In [ ]:
# Ejecutar una consulta con trazabilidad completa
print("🔭 Consulta con trazabilidad Langfuse:\n")

traced_result = orchestrator_with_langfuse.route_and_answer(
    question="¿Cuál es el proceso de onboarding para nuevos empleados?",
    session_id="demo-session-001",
)

print_result(traced_result)

if langfuse_client:
    print("📡 Traza enviada a Langfuse. Verifica en:", LANGFUSE_HOST)

In [ ]:
# Ejemplo de cómo crear una traza manual con spans detallados
if langfuse_client:
    print("📊 Demo de traza manual con spans:\n")
    
    # Crear traza padre
    trace = langfuse_client.trace(
        name="demo_manual_trace",
        input={"question": "¿Cuánto cuesta el plan de seguro médico?"},
        tags=["demo", "manual", "hr"],
        session_id="demo-manual-001",
    )
    
    # Span de routing
    routing_span = trace.span(name="routing", input={"question": "¿Cuánto cuesta el plan de seguro médico?"})
    time.sleep(0.1)  # Simular latencia
    routing_span.end(output={"domain": "hr", "confidence": 0.92})
    
    # Span de RAG
    rag_span = trace.span(name="hr_rag_pipeline")
    time.sleep(0.1)
    rag_span.end(output={"chunks_retrieved": 4})
    
    # Generation
    gen = trace.generation(
        name="hr_generation",
        model="gpt-4o-mini",
        input=[{"role": "user", "content": "¿Cuánto cuesta el seguro?"}],
    )
    gen.end(output="El seguro médico es completamente cubierto por TechCorp...")
    
    # Score
    langfuse_client.score(
        trace_id=trace.id,
        name="demo_quality",
        value=0.9,
        comment="Respuesta de demo con alta calidad",
    )
    
    langfuse_client.flush()
    print(f"✅ Traza manual enviada. ID: {trace.id}")
    print(f"   Ver en: {LANGFUSE_HOST}")
else:
    print("⚠ Langfuse no activo — saltando demo de traza manual")

---
## ⭐ Sección 7 (BONUS): Agente Evaluador

El evaluador usa el patrón **LLM-as-Judge** para calificar automáticamente
las respuestas del sistema en tres dimensiones:

| Métrica | Descripción |
|---------|-------------|
| **correctness** | ¿La información es factualmente correcta? |
| **relevance** | ¿La respuesta aborda la pregunta? |
| **completeness** | ¿La respuesta es suficientemente completa? |
| **routing_accuracy** | ¿El agente correcto respondió? |

Los scores se registran en Langfuse via **Score API** para análisis histórico.

In [ ]:
from evaluator import MultiAgentEvaluator

# Inicializar evaluador
evaluator = MultiAgentEvaluator(
    openai_client=openai_client,
    langfuse_client=langfuse_client,
    judge_model=CHAT_MODEL,
)

print("✅ Evaluador inicializado")
print(f"   Modelo juez: {evaluator.judge_model}")
print(f"   Langfuse: {'activo' if langfuse_client else 'inactivo'}")

In [ ]:
# Evaluación de una respuesta individual
print("🔍 Evaluación de respuesta individual:\n")

sample_result = orchestrator_with_langfuse.route_and_answer(
    "¿Cuál es el proceso para solicitar un adelanto de sueldo?"
)

eval_result = evaluator.evaluate(
    result=sample_result,
    expected_domain="hr",
)

print(f"Pregunta: {eval_result['question']}")
print(f"\n📊 Scores:")
for metric, score in eval_result['scores'].items():
    if score is not None:
        bar = '█' * int(score * 10) + '░' * (10 - int(score * 10))
        print(f"   {metric:20s}: {bar} {score:.2f}")

print(f"\n💬 Comentario general: {eval_result['overall_comment']}")
print(f"\n🔍 Razones detalladas:")
for metric, reason in eval_result['reasons'].items():
    print(f"   {metric}: {reason}")

In [ ]:
# Evaluación batch de un subconjunto de test queries
print("🚀 Evaluación batch de test queries...\n")

# Seleccionar primeras 8 queries para demo (reducir costos en demo)
eval_queries = test_queries[:8]

# Obtener respuestas para evaluarlas
eval_results = []
for tq in eval_queries:
    result = orchestrator_with_langfuse.route_and_answer(tq["query"])
    eval_results.append(result)

# Ejecutar evaluación batch
batch_summary = evaluator.batch_evaluate(
    results=eval_results,
    test_queries=eval_queries,
)

In [ ]:
# Visualizar resultados de la evaluación batch
print("📊 Resultados Detallados por Query:\n")
print(f"{'Query':<50} {'Dom':<8} {'Corr':>6} {'Rel':>6} {'Comp':>6} {'Overall':>8}")
print("-" * 90)

for ev in batch_summary["evaluations"]:
    query_short = ev["question"][:47] + "..."
    domain = ev["predicted_domain"][:6]
    scores = ev["scores"]
    corr = f"{scores.get('correctness', 0):.2f}"
    rel = f"{scores.get('relevance', 0):.2f}"
    comp = f"{scores.get('completeness', 0):.2f}"
    overall = f"{scores.get('overall', 0):.3f}"
    routing_ok = "✅" if scores.get("routing_accuracy") == 1.0 else "❌" if scores.get("routing_accuracy") == 0.0 else "?"
    print(f"{query_short:<50} {domain:<8} {corr:>6} {rel:>6} {comp:>6} {overall:>8} {routing_ok}")

print("-" * 90)
agg = batch_summary["aggregate_scores"]
print(f"{'PROMEDIO':<58} {agg['correctness']:>6.2f} {agg['relevance']:>6.2f} {agg['completeness']:>6.2f} {agg['overall']:>8.3f}")

In [ ]:
# Guardar resultados de evaluación
eval_output_path = OUTPUTS_DIR / "evaluation_results.json"

with open(eval_output_path, "w", encoding="utf-8") as f:
    json.dump(batch_summary, f, ensure_ascii=False, indent=2)

print(f"✅ Resultados de evaluación guardados en: {eval_output_path}")

# Flush Langfuse para asegurar que todos los datos fueron enviados
if langfuse_client:
    langfuse_client.flush()
    print("✅ Datos enviados a Langfuse")

In [ ]:
# Resumen final del sistema
print("\n" + "="*60)
print("🎉 RESUMEN DEL SISTEMA MULTI-AGENTE TECHCORP")
print("="*60)
print(f"\n📚 Base de Conocimiento:")
for domain, vs in vector_stores.items():
    print(f"   {domain.upper()}: {len(vs['chunks'])} chunks indexados")

print(f"\n🎯 Routing Accuracy (test set completo): {routing_accuracy:.1%}")

if batch_summary:
    agg = batch_summary["aggregate_scores"]
    print(f"\n📊 Calidad de Respuestas (LLM-as-Judge):")
    print(f"   Correctness:  {agg['correctness']:.2f}/1.0")
    print(f"   Relevance:    {agg['relevance']:.2f}/1.0")
    print(f"   Completeness: {agg['completeness']:.2f}/1.0")
    print(f"   Overall:      {agg['overall']:.3f}/1.0")

print(f"\n🔭 Observabilidad: {'Langfuse activo ✅' if langfuse_client else 'Langfuse no configurado ⚠'}")
print("="*60)